# StatBridge 골든셋 v1 — 전체 검증 노트북

`eval/statbridge-golden/v1/`의 골든셋 **전체(dev 30 · test 42 · holdout 18)** 를 자동 점검합니다. 표준 라이브러리만 사용하며, 시연·그래프 셀은 없습니다. 커널에서 **Run All** 하세요.

**검증 항목**
1. 개수·층화 (manifest 대조)
2. 구조·정합성 (스키마, id 유일)
3. fixture 무결성 (존재·해시·스냅샷·원본 CSV 해시·기간)
4. 누수 (holdout 키 제한, 질의 중복·재등장)
5. 우주·참조 (표 ∈ 349 카탈로그, graph 계열 ⊆ 정답 집합)
6. 해석 주장 진리검사 (모든 required_claims를 fixture와 대조)
7. 채점 자기일관성 (gold-echo → 전건 correct)

각 검증은 `PASS`/`FAIL`/`SKIP`으로 출력됩니다. holdout에는 정답이 없어(질문만) 6·7은 dev/test 대상입니다.

## 0. 설정

In [ ]:
from pathlib import Path
import sys, json, hashlib, collections, tempfile


def find_root(start):
    p = Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "eval/statbridge-golden/v1/scripts/score.py").is_file():
            return cand
    raise FileNotFoundError("StatBridge repo root not found (리포 안에서 실행하세요)")


ROOT = find_root(Path.cwd())
V1 = ROOT / "eval/statbridge-golden/v1"
sys.path.insert(0, str(V1 / "scripts"))
import validate as V   # scripts/validate.py
import score as S      # scripts/score.py

MANIFEST = json.loads((V1 / "manifest.json").read_text(encoding="utf-8"))
CATALOG_PATH = ROOT / "data/kosis/hankook_tables.json"
CATALOG = ({t["tbl_id"] for t in json.loads(CATALOG_PATH.read_text(encoding="utf-8"))["tables"]}
           if CATALOG_PATH.is_file() else None)

print("ROOT:", ROOT)
print("V1  :", V1)
print("카탈로그 표:", len(CATALOG) if CATALOG else "SKIP (data/kosis/hankook_tables.json 없음)")


## 1. 데이터 로드

In [ ]:
def load_split(name):
    p = V1 / f"cases/{name}.jsonl"
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]


dev = load_split("dev")
test = load_split("test")
holdout = load_split("holdout_queries")
labeled = dev + test


def type_of(r):
    g = r.get("gold")
    if not g:
        return "holdout"
    st = g["discovery"]["expected_status"]
    if st == "clarify":
        return "clarify"
    if st == "no_match":
        return "no_match"
    if g.get("graph", {}).get("layout") == "combined" and g.get("interpretation", {}).get("status") == "labeled":
        return "multi"
    if r["input"].get("prior_turns"):
        return "followup"
    return "single"


results = {}
def check(name, ok, detail=""):
    results[name] = (bool(ok), detail)

print("dev:", len(dev), "| test:", len(test), "| holdout:", len(holdout), "| labeled:", len(labeled))


## 검증 1. 개수·층화 (manifest 대조)

holdout은 정답이 없어 유형(type)을 알 수 없으므로, 층화는 dev/test만 대조합니다.

In [ ]:
TARGET = MANIFEST["targets"]; MATRIX = MANIFEST["type_matrix"]
bad = []
for name, got in (("dev", len(dev)), ("test", len(test)), ("holdout", len(holdout))):
    if got != TARGET[name]:
        bad.append(f"{name} {got}!={TARGET[name]}")
total = len(dev) + len(test) + len(holdout)
if total != TARGET["total"]:
    bad.append(f"total {total}!={TARGET['total']}")
check("1 개수", not bad, "; ".join(bad))

cnt_d = collections.Counter(type_of(r) for r in dev)
cnt_t = collections.Counter(type_of(r) for r in test)
m_bad = []
for typ, row in MATRIX.items():
    if cnt_d.get(typ, 0) != row["dev"]:
        m_bad.append(f"dev/{typ} {cnt_d.get(typ, 0)}!={row['dev']}")
    if cnt_t.get(typ, 0) != row["test"]:
        m_bad.append(f"test/{typ} {cnt_t.get(typ, 0)}!={row['test']}")
check("1 층화(dev/test)", not m_bad, "; ".join(m_bad))
print("counts:", dict(cnt_d), dict(cnt_t))
print({k: ("PASS" if v[0] else "FAIL " + v[1]) for k, v in results.items() if k.startswith("1")})


## 검증 2. 구조·정합성

In [ ]:
errs2 = []
for r in labeled:
    errs2 += V.validate_record(r)
ids = [r.get("id", "?") for r in labeled] + [r.get("id", "?") for r in holdout]
if len(ids) != len(set(ids)):
    errs2.append("duplicate ids across splits")
check("2 구조·정합성", not errs2, "; ".join(errs2[:5]))
print("구조 오류:", len(errs2))
for e in errs2[:30]:
    print(" -", e)


## 검증 3. fixture 무결성

존재·`fixture_sha256`·`snapshot_id`·원본 CSV 해시·기간 정렬/중복 + `numeric`/`graph` fixture 일치.

In [ ]:
errs3 = []
for r in labeled:
    errs3 += V.validate_fixtures(r, V1)
for r in labeled:
    g = r["gold"]; nu = g.get("numeric", {}); gr = g.get("graph", {})
    if nu.get("status") == "labeled" and gr.get("status") == "labeled" and nu.get("fixture") != gr.get("fixture"):
        errs3.append(f"{r['id']}: numeric/graph fixture differ")
check("3 fixture 무결성", not errs3, "; ".join(errs3[:5]))
print("fixture 오류:", len(errs3), "| fixture 파일:", len(list((V1 / "fixtures").glob("*.json"))))
for e in errs3[:30]:
    print(" -", e)


## 검증 4. 누수 (holdout 키 제한 · 질의 중복/재등장)

In [ ]:
errs4 = V.validate_holdout(holdout)


def qdupes(rows):
    seen = collections.Counter((r.get("query") or r.get("input", {}).get("query")) for r in rows)
    return [q for q, c in seen.items() if c > 1 and q]


for name, rows in (("dev", dev), ("test", test), ("holdout", holdout)):
    d = qdupes(rows)
    if d:
        errs4.append(f"{name} 중복 질의 {len(d)}")
lq = {(r.get("query") or r["input"]["query"]) for r in labeled}
overlap = lq & {r["query"] for r in holdout}
if overlap:
    errs4.append(f"holdout 질의가 dev/test에 재등장 {len(overlap)}")
check("4 누수", not errs4, "; ".join(errs4[:5]))
print("누수/중복 문제:", errs4 or "없음")


## 검증 5. 우주·참조 (표 ∈ 349 카탈로그, graph 계열 ⊆ 정답 집합)

In [ ]:
if CATALOG is None:
    check("5 우주·참조", True, "SKIP (카탈로그 없음)")
    print("SKIP: data/kosis/hankook_tables.json 없음")
else:
    errs5 = []
    for r in labeled:
        g = r["gold"]; d = g.get("discovery", {})
        acc = set(d.get("acceptable_table_ids", []))
        for tid in acc | set(d.get("required_set", [])) | set(d.get("forbidden_table_ids", [])):
            if tid not in CATALOG:
                errs5.append(f"{r['id']}: 표 {tid} 카탈로그에 없음")
        for s in g.get("graph", {}).get("series", []):
            if s["table_id"] not in acc:
                errs5.append(f"{r['id']}: graph 계열 {s['table_id']} ∉ acceptable")
    check("5 우주·참조", not errs5, "; ".join(errs5[:5]))
    print("참조 오류:", len(errs5))
    for e in errs5[:30]:
        print(" -", e)


## 검증 6. 해석 주장 진리검사

모든 `required_claims`를 fixture 데이터와 대조합니다. `direction`/`subperiod`/`change`는 기간 범위(하위기간 포함), `level`은 특정 시점, `extreme`은 전 구간 max/min, `comparison`은 `metric`(change/gap/수준)별로 검사합니다.

In [ ]:
def parse_period(p):
    if isinstance(p, str) and "~" in p:
        a, b = p.split("~")
        return a.strip().replace("-", ""), b.strip().replace("-", "")
    return ((p, p) if p else (None, None))


def series_by_name(slist, name):
    if not name:
        return slist[0] if len(slist) == 1 else None
    for s in slist:
        if s.get("label") == name:
            return s
    for s in slist:
        if name and (name in s.get("label", "") or s.get("label", "") in name):
            return s
    return None


def points_in(s, start, end):
    return [pt for pt in s["points"] if (start is None or pt["period"] >= start) and (end is None or pt["period"] <= end)]


def check_claim(c, slist):
    t = c["type"]
    if t in ("direction", "subperiod"):
        s = series_by_name(slist, c.get("series"))
        st, en = parse_period(c.get("period"))
        pts = points_in(s, st, en)
        if len(pts) < 2:
            return False, "기간 부족"
        d = pts[-1]["value"] - pts[0]["value"]
        return (d >= 0) == (c.get("expected") == "up"), f"d={round(d, 4)} want={c.get('expected')}"
    if t == "change":
        s = series_by_name(slist, c.get("series"))
        st, en = parse_period(c.get("period"))
        pts = points_in(s, st, en)
        d = pts[-1]["value"] - pts[0]["value"]
        return abs(d - c["expected_delta"]) <= c.get("tolerance", 0) + 1e-9, f"d={round(d, 4)} want={c['expected_delta']}"
    if t == "level":
        s = series_by_name(slist, c.get("series"))
        v = next((pt["value"] for pt in s["points"] if pt["period"] == c["period"]), None)
        return (v is not None and abs(v - c["expected_value"]) <= c.get("tolerance", 0) + 1e-9), f"v={v} want={c['expected_value']}"
    if t == "extreme":
        s = series_by_name(slist, c.get("series"))
        pts = s["points"]
        pick = max(pts, key=lambda x: x["value"]) if c.get("subtype") == "max" else min(pts, key=lambda x: x["value"])
        ok = pick["period"] == c.get("period") and abs(pick["value"] - c.get("value", pick["value"])) < 1e-9
        return ok, f"pick={pick['period']}:{pick['value']} want={c.get('period')}:{c.get('value')}"
    if t == "comparison":
        metric = c.get("metric")
        a, b = c["series"]
        sa, sb = series_by_name(slist, a), series_by_name(slist, b)
        if metric == "change":
            da = sa["points"][-1]["value"] - sa["points"][0]["value"]
            db = sb["points"][-1]["value"] - sb["points"][0]["value"]
            return (da > db) == (c["expected"] == "A>B"), f"da={round(da,3)} db={round(db,3)} want={c['expected']}"
        if metric == "gap":
            p0, p1 = c["periods"]

            def gv(px, s):
                return next(pt["value"] for pt in s["points"] if pt["period"] == px)
            d = (gv(p1, sa) - gv(p1, sb)) - (gv(p0, sa) - gv(p0, sb))
            return abs(d - c["expected_delta"]) <= c.get("tolerance", 0) + 1e-9, f"gapd={round(d,3)} want={c['expected_delta']}"
        st, en = parse_period(c.get("period"))
        pa, pb = points_in(sa, st, en), points_in(sb, st, en)
        if c["expected"] == "A>B":
            ok = all(x["value"] > y["value"] for x, y in zip(pa, pb))
        else:
            ok = all(x["value"] < y["value"] for x, y in zip(pa, pb))
        return ok, "level compare"
    if t == "turning_point":
        s = series_by_name(slist, c.get("series"))
        return any(pt["period"] == c.get("period") for pt in s["points"]), "soft: 시점 존재"
    return True, "skip"


errs6 = []
nclaims = 0
for r in labeled:
    iv = r["gold"].get("interpretation", {})
    if iv.get("status") != "labeled":
        continue
    fx = json.loads((V1 / r["gold"]["graph"]["fixture"]).read_text(encoding="utf-8"))["series"]
    for c in iv.get("required_claims", []):
        nclaims += 1
        ok, reason = check_claim(c, fx)
        if not ok:
            errs6.append(f"{r['id']} {c.get('id')} {c['type']}: {reason}")
check("6 주장 진리", not errs6, "; ".join(errs6[:5]))
print("검사 주장:", nclaims, "| 거짓:", len(errs6))
for e in errs6[:30]:
    print(" -", e)


## 검증 7. 채점 자기일관성 (gold-echo → 전건 correct)

골든셋 정답을 그대로 예측으로 넣으면 채점기는 전건 `correct`여야 합니다. (holdout은 정답이 없어 제외)

In [ ]:
def gold_echo(r):
    G = r["gold"]; d = G["discovery"]; g = G.get("graph", {})
    fx = json.loads((V1 / g["fixture"]).read_text(encoding="utf-8")) if g.get("status") == "labeled" else {"series": []}
    s = G.get("slots", {}); c = G.get("concepts", {}); cl = G.get("clarification", {})
    pl = G.get("plan", {}); e = G.get("e2e_kpi", {})
    return {"id": r["id"], "status": d.get("expected_status"), "table_ids": d.get("acceptable_table_ids", []),
            "slots": {k: s[k] for k in ("metric", "metrics", "target", "frequency", "basis", "period") if k in s},
            "concepts": {"concepts": c.get("acceptable_concepts", [])},
            "clarification": {"asked": cl.get("need_clarification", False), "options": cl.get("acceptable_options", [])},
            "plan": {"tool_sequences": pl.get("acceptable_tool_sequences", [])},
            "e2e": {"pass": e.get("pass")},
            "graph": {"chart_type": g.get("chart_type"), "layout": g.get("layout"), "series": fx["series"]},
            "numeric": {"series": fx["series"]},
            "interpretation": {"claims": G.get("interpretation", {}).get("required_claims", [])}}


errs7 = []
for split, rows in (("dev", dev), ("test", test)):
    tmp = Path(tempfile.mkdtemp())
    pp = tmp / "p.jsonl"
    pp.write_text("".join(json.dumps(gold_echo(r), ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
    sm = S.score(V1 / f"cases/{split}.jsonl", pp, out_dir=tmp)["summary"]
    if sm["correct"] != sm["n"] or sm["forbidden_hit"] or sm["mismatch"]:
        errs7.append(f"{split}: {sm}")
check("7 채점 자기일관성", not errs7, "; ".join(errs7))
print("자기일관성:", "dev/test 전건 correct" if not errs7 else errs7)


## 커버리지 · 검수 현황

In [ ]:
LAYERS = V.GOLD_KEYS
print("계층별 labeled 수")
print(f"{'split':8}" + "".join(f"{l[:7]:>9}" for l in LAYERS))
for name, rows in (("dev", dev), ("test", test)):
    vals = [collections.Counter(r["gold"].get(l, {}).get("status") for r in rows).get("labeled", 0) for l in LAYERS]
    print(f"{name:8}" + "".join(f"{v:>9}" for v in vals))
appr = sum(r["review"]["human_approved"] for r in labeled)
print(f"\nhuman_approved: {appr}/{len(labeled)}  (검수 전이면 0)")
print("fixture 파일:", len(list((V1 / "fixtures").glob("*.json"))))


## 종합 결과

In [ ]:
print("=" * 64)
allok = True
for name in sorted(results):
    ok, detail = results[name]
    tag = "PASS" if ok else "FAIL"
    print(f"{tag:4}  {name}" + ("" if ok else f"   [{detail}]"))
    allok = allok and ok
print("=" * 64)
print("종합:", "ALL PASS" if allok else "일부 FAIL — 위 항목을 확인하세요")


## 한계

- 해석의 **환각률(unsupported-claim rate)** 과 **coverage**, 자연어→그래프 전 과정(`graph_e2e`) 채점은 아직 보류입니다.
- `turning_point` 검사는 시점 존재만 보는 soft 검사입니다.
- 카탈로그 파일(`data/kosis/hankook_tables.json`)이나 원본 CSV(`data/tables/`)가 없으면 해당 검사는 SKIP됩니다.
- holdout은 정답이 없어(질문만) 6·7 검증 대상이 아닙니다. holdout 정답은 비공개 저장소에 둡니다.